# TD 3 — Nettoyer un vrai fichier

**Analyse des données — L3 Économie**

Le salaire moyen de ce fichier vaut **67 745 euros par mois**. Personne ne l'a mal calculé.

Votre travail aujourd'hui : trouver pourquoi, et dire ce qu'il vaut réellement. Vous le ferez en douze questions et une demi-douzaine de décisions, dont chacune devra être écrite et chiffrée.

**Le livrable n'est pas un fichier propre.** C'est le **journal des décisions**, avec l'effet de chacune sur le résultat.

## Ce dont vous avez besoin

Le fichier `cm03-salaires.csv`, celui de la séance 3, et son **dictionnaire des variables**. Ouvrez le dictionnaire maintenant, dans un onglet à côté. Vous y reviendrez à chaque partie, et aucune question ne se répond sans lui.

> *Exécution → Tout exécuter* avant de démarrer. Les cellules marquées `TODO` sont à compléter ; chaque `________` se remplace par une seule expression.

In [ ]:
import pandas as pd
import numpy as np

BASE = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
        "analyse_des_donnees/main/data/")

brut = pd.read_csv(
    BASE + "cm03-salaires.csv",
    encoding="latin-1", sep=";", decimal=",", thousands=" ",
    dtype={"IDENT": str, "SECTEUR": str, "REGION": str}
)
df = brut.copy()          # on ne touche jamais a `brut`

print(brut.shape)
brut.head()

## Partie 1 — Le journal, ouvert dès le départ

Un journal rempli de mémoire à la fin de la séance est faux. Celui-ci s'ouvre avant la première décision.

In [ ]:
journal = []


def noter(etape, decision, justification, data):
    journal.append({
        "etape": etape,
        "decision": decision,
        "lignes": len(data),
        "salaire_moyen": round(data["SALAIRE_NET"].mean(), 2),
        "obs_salaire": int(data["SALAIRE_NET"].notna().sum()),
        "justification": justification,
    })
    return pd.DataFrame(journal)


noter("Chargement", "---", "Fichier brut", df)

**Question 1.** Notez ici le salaire moyen initial et le nombre de lignes. Ce sont vos points de comparaison pour tout le reste du TD.

Pourquoi ouvrir le journal maintenant plutôt qu'à la fin de la séance ?

*Votre réponse :*

In [ ]:
# TODO : combien de lignes sont strictement identiques a une autre ?
print(df.________().sum(), "lignes identiques partout")

# TODO : retirez-les, puis notez la decision
df = df.________()
noter("Doublons techniques", "Suppression", "Lignes identiques partout", df)

Le fichier perd quelques lignes et la moyenne ne bouge presque pas. L'étape était nécessaire, son effet est négligeable : le journal permet de constater les deux.

La partie 5 expliquera pourquoi ce n'est **pas** la même chose que des identifiants répétés.

## Partie 2 — Les manquants qui n'en ont pas l'air

In [ ]:
# Ce que pandas voit
print(df.isna().sum().sort_values(ascending=False).head(4))

# Ce qu'il ne voit pas
print()
print(df["SALAIRE_NET"].value_counts().head(4))

Deux colonnes ont des manquants visibles. `SALAIRE_NET` n'en a aucun, et trois de ses valeurs reviennent des centaines de fois.

**Ouvrez le dictionnaire.** Remplissez le tableau suivant avant de coder.

| Colonne | Non-réponse | Sans objet | Censure |
|---|---|---|---|
| `SALAIRE_NET` | | | |
| `PRIME` | | | |
| `HEURES` | | | |
| `DIPLOME` | | | |
| `TEMPS` | | | |
| `CONTRAT` | | | |
| `MOTIF_ABS` | | | |
| `ANCIENNETE` | | | |

In [ ]:
# TODO : ecrivez la fonction. Elle convertit en numerique,
#        puis remplace les codes de non-reponse par un manquant.
def nettoyer(serie, codes_manquants):
    s = pd.to_numeric(serie, errors="________")
    return s.________(codes_manquants, np.nan)


# TODO : completez la liste des codes, colonne par colonne, depuis le dictionnaire.
CODES_NR = {
    "SALAIRE_NET": [________],
    "PRIME":       [________],
    "HEURES":      [________],
    "DIPLOME":     [________],
    "TEMPS":       [________],
    "CONTRAT":     [________],
}

for colonne, codes in CODES_NR.items():
    df[colonne] = nettoyer(df[colonne], codes)

print(df[list(CODES_NR)].isna().sum())
noter("Codes NR -> manquant", "Conversion", "Dictionnaire, 6 colonnes", df)

**Question 2.** De combien le salaire moyen a-t-il bougé ? Donnez l'écart en euros et en facteur multiplicatif.

Aucun message d'erreur n'est apparu, ni avant ni après. Que faut-il en retenir ?

*Votre réponse :*

### Une colonne ne suit pas la règle

`MOTIF_ABS` n'est pas dans la liste ci-dessus. Regardez pourquoi.

In [ ]:
print(df["MOTIF_ABS"].value_counts().sort_index())

**Question 3.** Sur cette colonne, quel code est la non-réponse ? Que signifie le code `9` ?

Combien de situations un `replace(9, np.nan)` appliqué par habitude détruirait-il ? La fonction `nettoyer` est-elle en cause ?

*Votre réponse :*

In [ ]:
# TODO : qui sont les postes dont le salaire n'est pas renseigne ?
#        On se place sur le champ salarie, c'est-a-dire hors STATUT 3.
champ = df.loc[df["STATUT"] != 3].copy()
champ["nr"] = brut.loc[champ.index, "SALAIRE_NET"] == ________

print("champ salarie :", len(champ), "postes, dont",
      int(champ["nr"].sum()), "sans salaire renseigne",
      f"({100 * champ['nr'].mean():.1f} %)")

# TODO : croisez la non-reponse avec trois variables du fichier
for v in [________, ________, ________]:
    print(f"\n--- {v} ---")
    print((100 * champ.groupby(v)["nr"].mean()).round(1))

**Question 4.** Décrivez ce que montre chacun des trois croisements. Y a-t-il une variable qui explique la non-réponse, et deux qui n'expliquent rien ?

**Question 5.** De quel type est cette non-réponse : complètement au hasard, au hasard conditionnellement, ou non au hasard ? Justifiez.

- Qu'est-ce que les données vous permettent d'affirmer ?
- Qu'est-ce qu'elles ne vous permettent pas d'affirmer, et pourquoi ?
- Si vous supprimez ces lignes, la moyenne sera-t-elle sur-estimée ou sous-estimée ? Et les inégalités ?

*Vos réponses :*

## Partie 3 — Zéro n'est pas manquant

In [ ]:
# TODO : combien de salaires valent exactement zero, et qui sont ces personnes ?
print((df["SALAIRE_NET"] == ________).sum(), "salaires nuls")
print()
print(df.loc[df["SALAIRE_NET"] == ________, "________"].value_counts())

In [ ]:
# TODO : la moyenne avec et sans ces lignes
moyenne_avec = df["SALAIRE_NET"].mean()
moyenne_sans = df.loc[df["________"] != ________, "SALAIRE_NET"].mean()

print("Avec les nuls :", round(moyenne_avec, 2))
print("Sans les nuls :", round(moyenne_sans, 2))
print("Ecart relatif :", round(100 * (moyenne_sans / moyenne_avec - 1), 1), "%")

**Question 6.** Que signifie la valeur zéro dans cette colonne ? Trois hypothèses sont plausibles a priori : non-salariés hors champ, non-réponse codée en zéro, ou personnes n'ayant pas travaillé. Laquelle est la bonne, et qu'est-ce qui vous permet de trancher ?

**Question 7.** La moyenne change de combien ? Quelle décision retenez-vous, et quelle question posez-vous en la retenant ? « Salaire moyen » et « revenu d'activité moyen » ne sont pas la même chose.

*Vos réponses :*

In [ ]:
champ = df.loc[df["STATUT"] != 3].copy()
noter("Non-salaries", "Exclusion", "Hors champ : pas de salaire", champ)

# Un controle : que devient le code "sans objet" d'ANCIENNETE ?
print("ANCIENNETE == -1 restants :", int((champ["ANCIENNETE"] == -1).sum()))
print()
print(champ["CONTRAT"].value_counts(dropna=False).sort_index())

Deux codes « sans objet » ont disparu d'eux-mêmes, dans `ANCIENNETE` et dans `CONTRAT`. Pourquoi ? Notez-le, c'est un bon signe : un champ bien défini nettoie plusieurs colonnes à la fois.

## Partie 4 — La censure haute

In [ ]:
salaire = champ["SALAIRE_NET"]

# TODO : les valeurs les plus frequentes, puis les quantiles hauts
print(salaire.________().head(3))
print()
print(salaire.quantile([0.90, 0.95, ________, ________]).round(2))

**Question 8.** Que remarquez-vous en comparant les quantiles à 99 % et à 99,9 % ? Que dit le dictionnaire sur cette valeur ?

*Votre réponse :*

In [ ]:
# TODO : le meme mecanisme sur une autre colonne
print(champ["________"].value_counts().sort_index().tail(6))

In [ ]:
def gini(x):
    """Indice de Gini sur un tableau de valeurs positives."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n


# TODO : calculez le Gini sur les salaires observes
print("Gini :", round(gini(salaire.________().values), 4))

**Question 9.** Cet indice de Gini est-il sur-estimé ou sous-estimé ? Le sens de l'erreur est connu sans ambiguïté : dites lequel et pourquoi.

Pouvez-vous dire **de combien** ? Justifiez votre réponse.

**Question 10.** Écrivez, en une phrase, ce que vous feriez figurer dans la note méthodologique d'un mémoire qui utiliserait ce Gini. La phrase doit mentionner le plafond, la part des observations concernées, et la nature de la borne obtenue.

*Vos réponses :*

## Partie 5 — Les doublons

In [ ]:
# On repart de `brut` : `df` a perdu ses doublons techniques en partie 1.
# TODO : trois comptages differents
print(brut.________().sum(), "lignes identiques partout")
print(brut.duplicated(subset=[________]).sum(), "identifiants repetes")
print(brut.duplicated(subset=[________, ________]).sum(), "couples repetes")

doublons = brut[brut.duplicated(subset=["IDENT"], keep=False)]
(doublons
 .sort_values(["IDENT", "NOPOSTE"])
 [["IDENT", "NOPOSTE", "STATUT", "TEMPS", "HEURES", "SALAIRE_NET", "SECTEUR"]]
 .head(6))

**Question 11.** Les deux premiers comptages ne mesurent pas la même chose. Expliquez la différence, puis dites quelle est la **clé** de ce fichier.

- Sur quoi diffèrent les deux lignes d'un même `IDENT` ?
- Que perdriez-vous en supprimant les identifiants répétés ?
- Si votre question portait sur les **personnes** et non sur les postes, que faudrait-il faire à la place ?

*Votre réponse :*

## Partie 6 — Le journal, et le test de robustesse

In [ ]:
final = champ.loc[champ["SALAIRE_NET"].notna()].copy()
noter("Salaires non renseignes", "Suppression",
      "Analyse restreinte aux salaires observes", final)

pd.DataFrame(journal)

**Question 12.** Commentez votre journal, ligne par ligne.

- Quelle étape était nécessaire et sans effet ?
- Quelle étape a changé le résultat d'un facteur ?
- La dernière ligne ne change pas la moyenne. Qu'est-ce qu'elle change alors ?

*Votre réponse :*

In [ ]:
# TODO : et si vous aviez impute au lieu de supprimer ?
alternatif = champ["SALAIRE_NET"].fillna(champ["SALAIRE_NET"].________())
retenu     = final["SALAIRE_NET"]

comparaison = pd.DataFrame({
    "conserver": [retenu.mean(), retenu.std(), gini(retenu.values), len(retenu)],
    "imputer":   [alternatif.mean(), alternatif.________(),
                  gini(alternatif.values), len(alternatif)],
}, index=["moyenne", "ecart-type", "Gini", "observations"])

comparaison.round(4)

**Question 13.** Comparez les quatre lignes du tableau.

- Laquelle bouge le moins ? Pourquoi est-ce précisément ce qui rend l'imputation trompeuse ?
- Que devient la dispersion, et pourquoi ?
- Que devient le Gini ? Formulez ce que l'imputation fait à l'inégalité mesurée.
- Le nombre d'observations annoncé change. Quelle conséquence sur les erreurs-types d'une régression faite ensuite ?

**Question 14.** Le test de robustesse ne cherche pas la bonne réponse : il n'y en a pas.

Identifiez **votre décision la plus discutable** du TD, et dites si votre conclusion y survit. Distinguez la conclusion sur la moyenne et celle sur les inégalités : elles ne se comportent pas pareil.

*Vos réponses :*

## Avant la séance 4

1. Terminez votre journal. Il doit comporter une ligne par décision, avec sa justification.
2. Vérifiez que le notebook s'exécute **de haut en bas sans erreur**.
3. Reprenez `MOTIF_ABS` et `CONTRAT` dans le dictionnaire, et écrivez pour chacune quels codes sont des non-réponses et lesquels sont du hors champ.

Aucune note n'est attribuée. En revanche, les questions de diagnostic du contrôle continu porteront sur les erreurs les plus fréquentes rencontrées ici.

---

Supports, données et corrigés : `stefaniamarcassa.github.io/analyse_des_donnees`